In [ ]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

In [ ]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql2025;TrustServerCertificate=True;Integrated Security=True"

In [ ]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='DataExposed')BEGIN
    ALTER DATABASE DataExposed SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE DataExposed;
END
GO
CREATE DATABASE DataExposed
GO
USE DataExposed
GO

In [ ]:
create master key encryption by password = 'BigDataClusters4ever!'

In [ ]:
sp_configure 'external rest endpoint enabled', 1;
RECONFIGURE WITH OVERRIDE

In [ ]:
CREATE TABLE [dbo].[Playlists](
	[YT_ID] [nvarchar](100) NOT NULL,
	[Name] [nvarchar](100) NOT NULL
) ON [PRIMARY]

In [ ]:
Start-Process https://www.youtube.com/playlist?list=PLlrxD0HtieHieV7Jls72yFPSKyGqycbZR

In [ ]:
INSERT INTO [Playlists] VALUES ('PLlrxD0HtieHieV7Jls72yFPSKyGqycbZR','Data Exposed')

In [ ]:
CREATE TABLE [dbo].[Videos](
	[id] [int] IDENTITY(1,1) NOT NULL,
	[Video] [nvarchar](50) NOT NULL,
	[Playlist] [nvarchar](50) NOT NULL,
	[Title] [nvarchar](500) NULL,
	[Description] [nvarchar](max) NULL,
	[PublishedAt] [datetime] NULL
) ON [PRIMARY] TEXTIMAGE_ON [PRIMARY]

In [ ]:
CREATE DATABASE SCOPED CREDENTIAL [https://www.googleapis.com/youtube/v3/]
WITH IDENTITY = 'HTTPEndpointQueryString', SECRET = '{"key":"XXX"}';

In [ ]:
DECLARE @url NVARCHAR(MAX);
DECLARE @response NVARCHAR(MAX);
SELECT TOP 1 @url=N'https://www.googleapis.com/youtube/v3/playlistItems?part=contentDetails&maxResults=500&playlistId='  + YT_ID FROM Playlists

EXEC Sp_invoke_external_rest_endpoint
            @credential = [https://www.googleapis.com/youtube/v3/],
            @method = 'GET',
            @url = @url,
            @response = @response OUTPUT;
SELECT top 3 value FROM   Openjson(Json_query(@response, '$.result.items')) A 

In [ ]:
TRUNCATE TABLE Videos;
SET NOCOUNT OFF
DECLARE @PlayList NVARCHAR(255);
DECLARE @url NVARCHAR(MAX);
DECLARE @response NVARCHAR(MAX);
DECLARE @nextPageToken NVARCHAR(200);

DECLARE playlist_cursor CURSOR FOR
SELECT YT_ID FROM playlists;

OPEN playlist_cursor;
FETCH NEXT FROM playlist_cursor INTO @PlayList;

WHILE @@FETCH_STATUS = 0
BEGIN
    SET @nextPageToken = 'none';

    WHILE len(isnull(@nextPageToken,'')) >= 4
    BEGIN
        SET @url = N'https://www.googleapis.com/youtube/v3/playlistItems?part=contentDetails&maxResults=500&playlistId=' 
                   + @PlayList + '&pageToken=' + ISNULL(nullif(@nextPageToken,'none'), '');
        EXEC Sp_invoke_external_rest_endpoint
            @credential = [https://www.googleapis.com/youtube/v3/],
            @method = 'GET',
            @url = @url,
            @response = @response OUTPUT;
        INSERT INTO Videos (Playlist, video)
        SELECT @PlayList AS Playlist,
               JSON_VALUE(value, '$.contentDetails.videoId') AS VideoID
        FROM OPENJSON(JSON_QUERY(@response, '$.result.items'), 'strict $');
        set @nextPageToken = ''
        SELECT @nextPageToken = [value]
        FROM OPENJSON(JSON_QUERY(@response, '$.result'), 'strict $')
        WHERE [key] = 'nextPageToken';
        
    END

    FETCH NEXT FROM playlist_cursor INTO @PlayList;
END

CLOSE playlist_cursor;
DEALLOCATE playlist_cursor;


In [ ]:
SELECT TOP 3 * FROM Videos

In [ ]:
SET NOCOUNT ON
DECLARE @VideoId NVARCHAR(50);
DECLARE @response NVARCHAR(MAX);
DECLARE @title NVARCHAR(MAX);
DECLARE @description NVARCHAR(MAX);
DECLARE @publishedAt datetime;
DECLARE @url NVARCHAR(MAX);

DECLARE video_cursor CURSOR FOR
SELECT Video FROM Videos WHERE Title IS NULL OR PublishedAt is NULL;

OPEN video_cursor;
FETCH NEXT FROM video_cursor INTO @VideoId;

WHILE @@FETCH_STATUS = 0
BEGIN
    SET @url = 'https://www.googleapis.com/youtube/v3/videos?part=snippet&id=' + @VideoId;
    SET @response = NULL;
    SET @title = NULL;
    SET @description = NULL;

    BEGIN TRY
        EXEC Sp_invoke_external_rest_endpoint
            @credential = [https://www.googleapis.com/youtube/v3/],
            @method = 'GET',
            @url = @url,
            @response = @response OUTPUT;
            
        IF JSON_VALUE(@response, '$.result.items[0].snippet.title') IS NOT NULL
        BEGIN
            SELECT 
                @title = JSON_VALUE(value, '$.snippet.title'),
                @description = JSON_VALUE(value, '$.snippet.description'),
                @publishedAt = JSON_VALUE(value, '$.snippet.publishedAt')
            FROM OPENJSON(JSON_QUERY(@response, '$.result.items'), 'strict $');
        END
    END TRY
    BEGIN CATCH
        PRINT 'Failed to parse or retrieve metadata for Video ID: ' + ISNULL(@VideoId, 'NULL');
    END CATCH

    UPDATE Videos
    SET Title = @title,
        Description = @description,
        publishedAt = @publishedAt
    WHERE Video = @VideoId;

    FETCH NEXT FROM video_cursor INTO @VideoId;
END

CLOSE video_cursor;
DEALLOCATE video_cursor;
SET NOCOUNT OFF

In [ ]:
DELETE FROM Videos WHERE Title IS NULL

In [ ]:
SELECT Title FROM Videos WHERE [Description] like '%weissman%'

We could now even get the transcription of each video...

In [ ]:
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'Here is the Title and Description of a YouTube Video. Give me a one sentence summary: ' + title + ' -' + [description]
FROM Videos WHERE [Description] like '%weissman%' AND Title like '%Big Data Clusters%'

SELECT @prompt 

Think of translations etc.!

In [ ]:
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'Here is the Title and Description of a YouTube Video. Give me a one sentence summary: ' + title + ' -' + [description]
FROM Videos WHERE [Description] like '%weissman%' AND Title like '%Big Data Clusters%'

SET @prompt = Replace(Replace(Replace(@prompt, Char(13), ''), Char(10), ''),'"','''')

DECLARE @response NVARCHAR(max)
DECLARE @model NVARCHAR(250) = N'mistral'
DECLARE @payload NVARCHAR(max) = N'{"model":"' + @model + '","prompt":"' + @prompt + '","stream": false }'

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://ai-gpu.lab.bwdemo.io:443/api/generate',
  @payload = @payload,
  @timeout = 230,
  @response = @response output;

SELECT value FROM   Openjson(Json_query(@response, '$.result')) A WHERE  [key] = 'response' 

We could loop over every video and store that summary in our table...

In [ ]:
CREATE EXTERNAL MODEL ollama
WITH (
    LOCATION = 'https://ai-gpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

In [ ]:
ALTER TABLE Videos
ADD embeddings VECTOR(768)

In [ ]:
SELECT TOP 1 AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama) FROM Videos

In [ ]:
UPDATE Videos SET embeddings = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama) where title + ' - ' + description is not null and embeddings is null

In [ ]:
SELECT title FROM Videos where title like '%SQL query performance%' or description like '%sql query performance%'

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'how can I improve my sql query performance?'
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT top 3 title,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM videos p 
WHERE embeddings is not null 
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'my SQL Query is super slow :('
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT top 3 title,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM videos p 
WHERE embeddings is not null 
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'gen AI'
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT top 3 title,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM videos p 
WHERE embeddings is not null AND title like '%MVP Edition%'
ORDER BY distance;